In [3]:
%load_ext google.colab.data_table

import sqlite3
import pandas as pd

# 1. Creamos la base de datos en memoria
conn = sqlite3.connect(':memory:')

# 2. Definimos las tablas (Schema)
queries_schema = [
    """
    CREATE TABLE conductores (
        id_conductor INT PRIMARY KEY,
        nombre TEXT,
        categoria TEXT
    );
    """,
    """
    CREATE TABLE rutas (
        id_ruta INT PRIMARY KEY,
        origen TEXT,
        destino TEXT,
        distancia_km DECIMAL
    );
    """,
    """
    CREATE TABLE viajes (
        id_viaje INTEGER PRIMARY KEY AUTOINCREMENT,
        id_conductor INT REFERENCES conductores(id_conductor),
        id_ruta INT REFERENCES rutas(id_ruta),
        consumo_l DECIMAL,
        alerta_ralenti INT
    );
    """
]

# 3. Insertamos datos de prueba (los mismos de antes)
queries_inserts = [
    "INSERT INTO conductores VALUES (1, 'Lucía Iacono', 'E1'), (2, 'Marcos Paz', 'E1');",
    "INSERT INTO rutas VALUES (101, 'Posadas', 'Eldorado', 200.0), (102, 'Posadas', 'Alem', 95.0);",
    "INSERT INTO viajes (id_conductor, id_ruta, consumo_l, alerta_ralenti) VALUES (1, 101, 25.5, 0), (2, 101, 30.2, 1), (1, 102, 12.1, 0);"
]

# Ejecutamos la creación y carga
for q in queries_schema + queries_inserts:
    conn.execute(q)

# 4. LA CONSULTA SQL FINAL (Feature Engineering)
sql_query = """
SELECT
    c.nombre AS Conductor,
    r.origen || ' -> ' || r.destino AS Trayectoria,
    ROUND(v.consumo_l / r.distancia_km, 3) AS Eficiencia_L_KM,
    v.alerta_ralenti AS Alerta_Motor_Encendido
FROM viajes v
JOIN conductores c ON v.id_conductor = c.id_conductor
JOIN rutas r ON v.id_ruta = r.id_ruta
ORDER BY Eficiencia_L_KM ASC;
"""

# 5. Pasamos el resultado a un DataFrame de Pandas
df_resultado = pd.read_sql_query(sql_query, conn)

# 6. MOSTRAMOS LA TABLA CON EL FORMATO INTERACTIVO Y ESTÉTICO
df_resultado

The google.colab.data_table extension is already loaded. To reload it, use:
  %reload_ext google.colab.data_table


,Conductor,Trayectoria,Eficiencia_L_KM,Alerta_Motor_Encendido
0,Lucía Iacono,Posadas -> Alem,0.127,0
1,Lucía Iacono,Posadas -> Eldorado,0.128,0
2,Marcos Paz,Posadas -> Eldorado,0.151,1
